# Task 3.2. Unsupervised histopathology clustering

**Цель:** сгруппировать изображения тканей без использования правильных классов и присвоить каждому тестовому изображению номер кластера.

## Что требуется по условию

- Вход: `dataset/task3_2/test_features.npz`.
- Внутри должен быть массив `images` формы `(N, 28, 28, 3)` и обычно массив `image_id`.
- На выходе: `outputs/task3_2_predictions.csv` с колонками `image_id,prediction`.
- `prediction` — произвольный номер кластера. Он **не обязан** совпадать с исходной меткой ткани.
- Метки классов нельзя использовать при построении признаков или обучении кластеризатора.

Это задача **кластеризации без учителя**: алгоритм ищет группы похожих изображений сам. Мы используем цветовые гистограммы, HOG (контуры/структура), LBP (микротекстура), затем PCA и K-Means. Для PathMNIST задаём 9 кластеров, поскольку исходный набор содержит 9 типов тканей. Номера кластеров всё равно условные.

## 1. Подготовка файлов

В Google Colab загрузите архив задания через панель Files слева и распакуйте его в текущую рабочую директорию так, чтобы путь был:

`dataset/task3_2/test_features.npz`

Если в архиве есть `train_features.npz`, положите его рядом. Он будет использован только как дополнительный **неразмеченный** набор для обучения K-Means. Если его нет, ноутбук кластеризует сам тестовый набор.

Можно также смонтировать Google Drive и скопировать папку `dataset` оттуда.

In [ ]:
# Если архив задания загружен в Colab, распакуйте его командой:
# !unzip -q /content/task3_2.zip -d /content/
#
# Проверьте, что файл на месте:
!ls -lah dataset/task3_2

## 2. Установка библиотек

`scikit-image` нужен для HOG/LBP, `scikit-learn` — для PCA и K-Means.

In [ ]:
!pip -q install scikit-image scikit-learn pandas

## 3. Построение признаков и кластеризация

Код ниже не читает и не использует метки тканей. Он проверяет структуру NPZ, строит признаки, обучает K-Means и сохраняет CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.pipeline import make_pipeline
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2hsv
from skimage.transform import resize
from skimage.filters import sobel
import warnings

SEED = 42
N_CLUSTERS = 9  # PathMNIST содержит 9 типов тканей; это число кластеров, не метки классов.
DATA_DIR = Path("dataset/task3_2")
OUT_PATH = Path("outputs/task3_2_predictions.csv")

def load_npz(path):
    data = np.load(path, allow_pickle=True)
    print(f"\\n{path}:")
    for k in data.files:
        arr = data[k]
        print(f"  {k}: shape={arr.shape}, dtype={arr.dtype}")
    if "images" not in data:
        raise KeyError(f"В {path} нет массива 'images'")
    images = data["images"]
    ids = data["image_id"] if "image_id" in data else (
        data["image_ids"] if "image_ids" in data else np.arange(len(images))
    )
    return images, np.asarray(ids)

def image_features(images):
    # Цветовые признаки + HOG + локальная текстура. Никакие тканевые метки не используются.
    feats = []
    for im in images:
        x = np.asarray(im, dtype=np.uint8)
        if x.ndim != 3 or x.shape[-1] != 3:
            raise ValueError(f"Ожидалось RGB-изображение HxWx3, получено {x.shape}")
        f = []
        # RGB/HSV-гистограммы сохраняют цветовые различия окраски тканей.
        xf = x.astype(np.float32) / 255.0
        hsv = rgb2hsv(xf)
        for channel, bins, rng in [
            (xf[:, :, 0], 8, (0, 1)), (xf[:, :, 1], 8, (0, 1)),
            (xf[:, :, 2], 8, (0, 1)), (hsv[:, :, 0], 12, (0, 1)),
            (hsv[:, :, 1], 8, (0, 1)), (hsv[:, :, 2], 8, (0, 1))
        ]:
            hist, _ = np.histogram(channel, bins=bins, range=rng, density=True)
            f.extend(hist.astype(np.float32))
        gray = np.mean(xf, axis=2)
        f.extend(hog(gray, orientations=9, pixels_per_cell=(4, 4),
                     cells_per_block=(2, 2), feature_vector=True).astype(np.float32))
        # LBP-гистограмма фиксирует микротекстуру.
        lbp = local_binary_pattern((gray * 255).astype(np.uint8), P=8, R=1, method="uniform")
        hist, _ = np.histogram(lbp, bins=np.arange(0, 11), density=True)
        f.extend(hist.astype(np.float32))
        # Небольшая downsample-карта цвета добавляет пространственную информацию.
        small = resize(xf, (7, 7), anti_aliasing=True, preserve_range=True)
        f.extend(small.reshape(-1).astype(np.float32))
        feats.append(f)
    return np.asarray(feats, dtype=np.float32)

def make_embedding(images):
    X = image_features(images)
    # Масштабирование + PCA: уменьшает шум и корреляции признаков.
    n_comp = min(64, X.shape[0] - 1, X.shape[1])
    if n_comp >= 2:
        reducer = make_pipeline(StandardScaler(), PCA(n_components=n_comp, whiten=True,
                                                       random_state=SEED))
        Z = reducer.fit_transform(X)
    else:
        Z = StandardScaler().fit_transform(X)
    return normalize(Z)

test_path = DATA_DIR / "test_features.npz"
if not test_path.exists():
    raise FileNotFoundError(
        f"Не найден {test_path}. В Colab сначала загрузите архив задания и распакуйте его "
        "так, чтобы существовал dataset/task3_2/test_features.npz"
    )

X_test_img, test_ids = load_npz(test_path)
train_path = DATA_DIR / "train_features.npz"

if train_path.exists():
    X_train_img, train_ids = load_npz(train_path)
    print("\\nИспользуем train_features только как неразмеченные данные для обучения кластеризатора.")
    all_images = np.concatenate([X_train_img, X_test_img], axis=0)
    Z_all = make_embedding(all_images)
    Z_test = Z_all[len(X_train_img):]
    Z_fit = Z_all[:len(X_train_img)]
    model = KMeans(n_clusters=N_CLUSTERS, n_init=30, random_state=SEED)
    model.fit(Z_fit)
    predictions = model.predict(Z_test)
else:
    warnings.warn("train_features.npz не найден: кластеризуем test-набор целиком.")
    Z_test = make_embedding(X_test_img)
    model = KMeans(n_clusters=N_CLUSTERS, n_init=30, random_state=SEED)
    predictions = model.fit_predict(Z_test)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission = pd.DataFrame({
    "image_id": test_ids,
    "prediction": predictions.astype(int)
})
assert len(submission) == len(X_test_img)
assert submission["image_id"].isna().sum() == 0
assert submission["prediction"].isna().sum() == 0
assert submission["image_id"].is_unique
submission.to_csv(OUT_PATH, index=False)

print("\\nРазмер результата:", submission.shape)
print("Кластеры:", submission["prediction"].value_counts().sort_index().to_dict())
print("Сохранено:", OUT_PATH)
display(submission.head())


## 4. Проверка результата

Файл `outputs/task3_2_predictions.csv` должен содержать ровно столько строк, сколько изображений в `test_features.npz`. Кластеры могут иметь номера `0..8`; это не классы тканей, а найденные группы.

Скачайте CSV из панели Files Colab. Для сдачи также сохраните этот ноутбук как `your-sollution/task3_2_solution.ipynb`. Рядом можно положить Python-скрипт `task3_2_solution.py`.

In [ ]:
from google.colab import files
files.download("outputs/task3_2_predictions.csv")